# 3.3 — データのクリーニングと監査記録

元データを保持したまま品質問題を検出し、規則・件数・処置・検証結果を記録できる分析用データを作ります。

## 導入

このNotebookでは、Moodle本文の概念を実際のデータとコードで確かめます。

## このレッスンの到達目標

- 原本、作業用、分析用のデータを分けて保持できる。
- 型変換失敗、欠損、表記ゆれ、範囲違反、項目間矛盾、重複を区別して検出できる。
- 複数の品質理由を残した確認対象表を作れる。
- 件数照合と監査記録により、分析用データが作られた過程を説明できる。

> **学習経路:** 必須：3.3.1〜3.3.6　／　統合練習：3.3.7


## 3.3.1 原本を保ち、品質問題を定義する

3.2では、条件に合う行を選びました。しかし、欠損や表記ゆれ、不可能な値を含むままでは、同じ抽出式でも意味のある結果になりません。そこで、まず元データを保存し、問題を数え、判定規則を決め、その規則に従って処置し、最後に件数と制約を再確認します。この順序が監査可能なクリーニングです。

In [ ]:
from pathlib import Path
import pandas as pd


def find_course_data(filename):
    """Find a course data file without depending on the notebook start folder."""
    roots = [Path.cwd(), *Path.cwd().parents, Path.home() / "work", Path("/opt/python-lab/course-materials")]
    checked = []
    for root in roots:
        for candidate in (root / "data" / filename, root / filename):
            candidate = candidate.expanduser()
            if candidate in checked:
                continue
            checked.append(candidate)
            if candidate.is_file():
                return candidate
    locations = "\n".join(f"- {candidate}" for candidate in checked)
    raise FileNotFoundError(f"Course data file {filename!r} was not found. Checked:\n{locations}")


data_file = find_course_data("learning-centres-practice.csv")
raw = pd.read_csv(data_file)
print("Loading:", data_file.resolve())
print("Rows:", len(raw), "Columns:", len(raw.columns))


## 3.3.2 原本・作業用・分析用データを分ける

欠損件数、データ型、カテゴリ値、数値範囲を先に表示します。`raw`は読み込んだ状態のまま残し、処理は`clean = raw.copy(deep=True)`から始めます。元データと処理後データを区別できなければ、何を変えたか検証できません。

In [ ]:
print(raw.dtypes)
print("Missing values:\n", raw.isna().sum())
print("District labels:", sorted(raw["district"].dropna().unique()))
print(raw[["registered", "attended", "completed", "material_cost"]].describe())

clean = raw.copy(deep=True)


## 3.3.3 型変換失敗と欠損を扱う

CSVの数値列に文字が混じると、列全体が文字列として読まれることがあります。`pd.to_numeric(..., errors="coerce")`は変換不能値を欠損値へ変えます。ただし、変換前から空だった値と、新たに変換できなかった値は別の問題です。変換前後のマスクを比較し、発生件数を記録します。

In [ ]:
numeric_columns = ["registered", "attended", "completed", "training_hours", "material_cost"]
conversion_failures = {}
for column in numeric_columns:
    before_missing = clean[column].isna()
    converted = pd.to_numeric(clean[column], errors="coerce")
    failed = converted.isna() & ~before_missing
    conversion_failures[column] = int(failed.sum())
    clean[column] = converted

print("Conversion failures:", conversion_failures)


### 欠損と0を混同しない

出席者数の空欄を0で埋めると、「未報告」を「出席者なし」に変えてしまいます。平均や割合も変わります。根拠がなければ推測で補完せず、`isna()`でフラグを作り、集計対象から除外するか、確認待ちとして隔離するかを記録します。

In [ ]:
missing_attended = clean["attended"].isna()
print("Missing attended:", int(missing_attended.sum()))
print(clean.loc[missing_attended, ["month", "centre_id", "attended", "completed"]])


## 3.3.4 表記を整え、元の値を残す

前後の空白や大文字・小文字の違いで、同じ地区が別カテゴリになることがあります。新しい列へ`strip()`と`title()`を適用し、変化した行を数えます。ただし、似ている語を自動的に同一視してはいけません。表記規則や対応表で同じ意味だと確認できる場合だけ統一します。

In [ ]:
clean["district_raw"] = clean["district"]
clean["district"] = clean["district"].astype("string").str.strip().str.title()
changed_district = clean["district_raw"].astype("string") != clean["district"]
print("Changed district labels:", int(changed_district.sum()))
print(clean.loc[changed_district, ["district_raw", "district"]].drop_duplicates())


## 3.3.5 範囲・項目間制約・重複を検査する

このデータでは、人数は0以上で、登録者数以上に出席者は存在せず、出席者数以上に修了者は存在しないと定義します。各規則を別のブールマスクにすると、どの規則に何件違反したか説明できます。欠損は不正値と同一視せず、別に数えます。

In [ ]:
negative_count = (clean[["registered", "attended", "completed"]] < 0).any(axis=1)
attendance_over_registered = clean["attended"].notna() & (clean["attended"] > clean["registered"])
completion_over_attendance = clean["completed"].notna() & clean["attended"].notna() & (clean["completed"] > clean["attended"])
negative_cost = clean["material_cost"].notna() & (clean["material_cost"] < 0)

print("Negative learner counts:", int(negative_count.sum()))
print("Attendance above registration:", int(attendance_over_registered.sum()))
print("Completion above attendance:", int(completion_over_attendance.sum()))
print("Negative material cost:", int(negative_cost.sum()))
print(clean.loc[completion_over_attendance, ["month", "centre_id", "registered", "attended", "completed"]])


### 業務キーで重複グループ全体を調べる

全列が同じかどうかだけでは、二重登録を見つけられないことがあります。この表では「センター・月・コース」を1記録と定義し、`duplicated(..., keep=False)`で重複グループの全行を表示します。同じセンターが別月に現れることは正当なので、キーの定義が先です。

In [ ]:
business_key = ["centre_id", "month", "course"]
duplicate_key = clean.duplicated(subset=business_key, keep=False)
print("Rows with duplicate business keys:", int(duplicate_key.sum()))
print(clean.loc[duplicate_key].sort_values(business_key))


## 3.3.6 確認対象と監査記録を残す

問題を見つけた直後に元の行を削除してはいけません。信頼できる原資料から修正できるなら修正し、表記規則が明確なら正規化し、根拠が足りなければ欠損または確認待ちとして扱います。ここでは不可能な修了者数を推測で直さず、分析対象フラグをFalseにします。`raw`と`clean`の行数は保持します。

In [ ]:
clean["analysis_ready"] = ~(
    missing_attended
    | negative_count
    | attendance_over_registered
    | completion_over_attendance
    | negative_cost
    | duplicate_key
)
analysis = clean.loc[clean["analysis_ready"]].copy()
print("Raw rows:", len(raw))
print("Retained clean rows:", len(clean))
print("Analysis-ready rows:", len(analysis))
print("Flagged rows:", int((~clean["analysis_ready"]).sum()))


### 問題理由を行単位の確認対象表へ残す

規則ごとの件数だけでは、原資料のどの行を確認すべきか分かりません。個別フラグは消さず、一定の順序で理由を連結し、元の識別列とともに確認対象表へ取り出します。一行が複数規則へ違反した場合も、最初の理由だけにせず該当理由をすべて残します。これにより、分析から外した件数と、確認担当者へ渡すレコードを同じ判定から作れます。


In [ ]:
issue_rules = [
    (missing_attended, "missing attended"),
    (completion_over_attendance, "completion above attendance"),
    (negative_cost, "negative material cost"),
    (duplicate_key, "duplicate business key"),
]
clean["issue"] = ""
for mask, label in issue_rules:
    clean.loc[mask, "issue"] = clean.loc[mask, "issue"] + label + "; "
clean["issue"] = clean["issue"].str.rstrip("; ")

verification_columns = ["month", "centre_id", "course", "issue"]
records_to_verify = (
    clean.loc[~clean["analysis_ready"], verification_columns]
    .sort_values(["month", "centre_id", "course"])
    .reset_index(drop=True)
)
records_to_verify


### 監査記録とassertで再検証する

監査記録はコードの代わりではなく、コードが行った判断の要約です。問題名、検出規則、影響件数、処置、処置後の残件を記録します。0件だった検査も、確認した証拠として残します。

In [ ]:
audit = pd.DataFrame([
    {"issue": "missing attended", "rule": "attended is missing", "affected": int(missing_attended.sum()), "action": "exclude from rate analysis; request review"},
    {"issue": "district spelling", "rule": "strip whitespace and title case", "affected": int(changed_district.sum()), "action": "normalise; preserve district_raw"},
    {"issue": "completion above attendance", "rule": "completed <= attended", "affected": int(completion_over_attendance.sum()), "action": "flag; do not guess replacement"},
    {"issue": "duplicate business key", "rule": "unique centre_id + month + course", "affected": int(duplicate_key.sum()), "action": "review duplicate group"},
])
audit


### 同じ制約と件数で再検証する

処置後のデータへ同じ制約を適用し、違反が残っていないことを確認します。さらに、元件数が「分析可能件数＋フラグ件数」と一致するか照合します。`assert`は期待が崩れた場所で処理を止め、翌月のデータ更新で規則が破られたことを知らせます。

In [ ]:
assert len(raw) == len(clean)
assert len(clean) == int(clean["analysis_ready"].sum()) + int((~clean["analysis_ready"]).sum())
assert not (analysis["completed"] > analysis["attended"]).any()
assert not analysis.duplicated(subset=business_key).any()
print("Validation passed")


## 3.3.7 統合練習：品質規則を別の表へ適用する

練習CSVについて、欠損した修了者数、登録者数を上回る出席者数、0以下の研修時間、負の教材費、業務キーの重複を検査してください。各マスクの件数を表示し、処置を推測で決めず、規則・件数・提案する処置を監査表にまとめます。最後に元件数との照合式を追加してください。

In [ ]:
# ここに応用練習の解答を書きます。


## まとめ

- 原本を保ったまま作業用コピーへ品質フラグを追加しました。
- 問題ごとに名前付き規則を作り、一行に複数ある理由も残しました。
- 分析対象と確認対象を分け、件数・制約・監査記録を再検証しました。

## 次のレッスンへ

分析に使える行と確認が必要な行を説明できる形で分けました。3.4では、分析用データを目的に合う粒度へ集計し、判断に使える指標と順位を作ります。

**学習時間の目安:** 約3時間
